<a href="https://colab.research.google.com/github/Numanur/llm-practice/blob/main/LLM_03_webtocolab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install -q fastapi uvicorn pyngrok nest-asyncio transformers accelerate

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer
from threading import Thread

model_id = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="cuda:0"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.eval()

print("Model loaded on:", model.device)

In [8]:
def generate_stream_chunks(user_prompt, max_new_tokens=300):
    messages = [
        {"role": "user", "content": user_prompt}
    ]

    model_inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )

    model_inputs = {k: v.to("cuda:0") for k, v in model_inputs.items()}

    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True
    )

    generation_kwargs = dict(
        **model_inputs,
        streamer=streamer,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    for new_text in streamer:
        yield new_text

In [9]:
for chunk in generate_stream_chunks("Explain retrieval augmented generation in simple words.", max_new_tokens=10):
    print(chunk, end="", flush=True)

Retrieval Augmented Generation (RAG

In [10]:
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/")
def home():
    return {"status": "ok", "message": "Streaming API is running"}

@app.post("/generate")
async def generate(request: Request):
    body = await request.json()
    prompt = body.get("prompt", "").strip()
    max_new_tokens = int(body.get("max_new_tokens", 300))

    if not prompt:
        return JSONResponse(
            status_code=400,
            content={"error": "Prompt is empty"}
        )

    def stream_response():
        try:
            for chunk in generate_stream_chunks(prompt, max_new_tokens=max_new_tokens):
                yield chunk
        except Exception as e:
            print("Generation error:", str(e))
            yield f"\n[ERROR] {str(e)}"

    return StreamingResponse(
        stream_response(),
        media_type="text/plain",
        headers={"Cache-Control": "no-cache"},
    )

In [2]:
import nest_asyncio
import uvicorn
from threading import Thread

nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = Thread(target=run_server, daemon=True)
server_thread.start()

print("Server started on port 8000")

Server started on port 8000


In [21]:
from pyngrok import ngrok

# ngrok.set_auth_token("3CenhdMC9xr7r8lyEVTrEJ9wwZ5_6CE9ZR6vzT8ZSxsaV6bVd")
print("ngrok auth configured.")

ngrok auth configured.


In [3]:
from pyngrok import ngrok

ngrok.kill()
tunnel = ngrok.connect(8000, bind_tls=True)
public_url = tunnel.public_url

print("Public URL:", public_url)
print("POST endpoint:", public_url + "/generate")

Public URL: https://worst-cogwheel-recolor.ngrok-free.dev
POST endpoint: https://worst-cogwheel-recolor.ngrok-free.dev/generate


In [ ]:
import requests

url = f"{public_url}/generate"

response = requests.post(
    url,
    json={
        "prompt": "Tell me in simple words what an embedding is.",
        "max_new_tokens": 120
    },
    stream=True
)

print("Status code:", response.status_code)
print("Streaming output:\n")

for chunk in response.iter_content(chunk_size=None, decode_unicode=True):
    if chunk:
        print(chunk, end="", flush=True)

In [ ]:
import requests

r = requests.options(
    public_url + "/generate",
    headers={
        "Origin": "http://localhost:5173",
        "Access-Control-Request-Method": "POST",
        "Access-Control-Request-Headers": "content-type",
    },
)

print("status:", r.status_code)
print("allow-origin:", r.headers.get("access-control-allow-origin"))
print("allow-methods:", r.headers.get("access-control-allow-methods"))
print("allow-headers:", r.headers.get("access-control-allow-headers"))
print("all headers:", dict(r.headers))